In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Terrain complexity assessment and transferability

## Summary

Demonstrate and visualize the application of T-RIX assessment of the transferability of wind assessments.

## Results

### Key results

- Examples are sunny-weather examples similar to the RIX demonstration.
- Sites are provided in some CRS, and for sampling from elevation data, transformed into the elevation data CRS.
  - The setup expects sites in a Cartesian CRS. If different, distances and angles are treated as if they were.

### Details

1. Requirements:
   - A map given as a `xarray.DataArray` given in its own coordinate system with coordinates `("y", "x")`.
   - If the map is given in a particular `CRS`, then keep this one, too.
   - A location on the map encoded in a `shapely.geometry.Point` object.
1. Parameters (metrics valid in metric CRS only):
   - `R_km`: Radius [km] of the disc of evaluation. TR6: 3.5km.
   - `dr_km`: Stepsize between points on each ray. `R_km` shall be a multiple of `dr_km`.
1. Free parameters:
   - `slope_critical`: Threshold [m/m] at which a slope is considered steep. This is no angle. TR6: 0.033.

## Remarks

1. Plots:
   - `plot_profile_and_steep_segments`: The slopes and its mask of steep slopes represent left-labelled intervals, and are plotted for simplicity at their label, i.e. at the left bound. The artifact is of visual nature.

## Imports and Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio

import phoibe.geography.crs
import phoibe.synthetic_data.fields
import phoibe.synthetic_data.sites
from phoibe.geography.complexity.rix import (
    NaNPolicy,
    RayGeometry,
    RayProfile,
    RayResult,
    RegularGridXYSampler,
    analyse,
    analyzer,
    config,
)

## Create scenario

## Create map and equip w/ CRS

In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
FREQ_X, FREQ_Y = 0.002, 0.002
BOUNDS = (11, 51.0, 14, 53.0)
CRS_TO = "32633"

R_km = 2
DR_km = 0.01
THETA = 90

eggbox_vanilla = (
    phoibe.synthetic_data.fields.make_eggbox_field(nx=NX, ny=NY, dx=DX, dy=DY, freq_x=FREQ_X, freq_y=FREQ_Y) * 1000
)
eggbox_gcs = phoibe.synthetic_data.fields.make_field_rio(da=eggbox_vanilla, bounds=BOUNDS, crs="4326", dtype="float")
eggbox_utm, meta_reproject = phoibe.geography.crs.reproject.reproject_rasterdata(
    da=eggbox_gcs, crs_to=CRS_TO, resolution=500, resampling=rasterio.warp.Resampling.nearest
)

meta_reproject

In [ ]:
eggbox_utm.rio.bounds()

### Create sites

In [ ]:
sites_utm = phoibe.synthetic_data.sites.make_sites(
    sites=7, bounds=eggbox_utm.rio.bounds(), buffer=3e3, seed=23, crs=eggbox_utm.rio.crs
).rename(columns={"name": "location_id"})

figure, axes = phoibe.geography.plot.plot_raster(da=eggbox_utm)
sites_utm.plot(color="r", ax=axes)
for _k, row in sites_utm.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)
sites_utm

In [ ]:
eggbox_utm.crs

## Analyse complexity

### In UTM w/ map in GCS

In [ ]:
cfg = config.ANALYZER_DEFAULTS.copy()
cfg["parameters"].update(
    {"crs": eggbox_gcs.rio.crs.to_authority(), "slope_critical": 0.01, "R_km": 17.0, "dr_km": 0.5, "n_angles": 32}
)
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(dem=eggbox_gcs, locations_site=sites_utm, locations_reference=sites_utm.iloc[5:, :])

In [ ]:
figure, axes = phoibe.geography.plot.plot_raster(da=eggbox_utm)
rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)
sites_utm.plot(color="b", ax=axes)
for _k, row in sites_utm.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

In [ ]:
def plot_polar(rix_directional, label=None, ax=None, figure=None):

    angles_radians = np.deg2rad(np.append(rix_directional.index, rix_directional.index[0]))
    values = rix_directional.values
    values = np.append(values, values[0])

    if ax is None:
        figure, ax = plt.subplots(subplot_kw={"projection": "polar"})

    ax.plot(angles_radians, values, label=label, linewidth=0.7)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_ylabel("RIX")

    return figure, ax

In [ ]:
rix_results.steep_segments.loc[rix_results.steep_segments.theta == 90.0]  # .geometry[331].xy
figure, ax = plt.subplots(subplot_kw={"projection": "polar"})
for key, row in rix_results.roses.iterrows():
    plot_polar(row, label=str(key), ax=ax, figure=figure)

ax.legend()
ax.set_title("RIX roses")

In [ ]:
display(rix_results.summary)

In [ ]:
rix_results.transferability

In [ ]:
rix_results.meta

### In UTM w/ map in UTM

In [ ]:
cfg = config.ANALYZER_DEFAULTS.copy()
cfg["parameters"].update(
    {"crs": eggbox_utm.rio.crs.to_authority(), "slope_critical": 0.01, "R_km": 17.0, "dr_km": 0.5, "n_angles": 32}
)
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(dem=eggbox_utm, locations_site=sites_utm, locations_reference=sites_utm.iloc[5:, :])

In [ ]:
figure, axes = phoibe.geography.plot.plot_raster(da=eggbox_utm)
rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)
sites_utm.plot(color="b", ax=axes)
for _k, row in sites_utm.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

### In GCS w/ map in GCS

No reasonable use case. Directions and distances can be distorted.

In [ ]:
sites_gcs = sites_utm.to_crs(crs=4326)
sites_gcs

In [ ]:
cfg = config.ANALYZER_DEFAULTS.copy()
cfg["parameters"].update(
    {"crs": eggbox_gcs.rio.crs.to_authority(), "slope_critical": 1e3, "R_km": 2e-4, "dr_km": 1e-5, "n_angles": 32}
)
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(dem=eggbox_gcs, locations_site=sites_gcs, locations_reference=sites_gcs.iloc[5:, :])

In [ ]:
rix_results.summary

In [ ]:
figure, axes = phoibe.geography.plot.plot_raster(da=eggbox_gcs)
rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)
sites_gcs.plot(color="b", ax=axes)
for _k, row in sites_gcs.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

### No CRS

In [ ]:
bounds = (
    float(eggbox_vanilla.x.min()),
    float(eggbox_vanilla.y.min()),
    float(eggbox_vanilla.x.max()),
    float(eggbox_vanilla.y.max()),
)
sites_vanilla = phoibe.synthetic_data.sites.make_sites(sites=7, bounds=bounds, buffer=200, seed=23, crs=None).rename(
    columns={"name": "location_id"}
)
sites_vanilla

In [ ]:
cfg = config.ANALYZER_DEFAULTS.copy()
cfg["parameters"].update({"crs": None, "slope_critical": 1, "R_km": 0.5, "dr_km": 0.01, "n_angles": 32})
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(
    dem=eggbox_vanilla, locations_site=sites_vanilla, locations_reference=sites_vanilla.iloc[5:, :]
)

In [ ]:
rix_results.summary

In [ ]:
figure, axes = phoibe.geography.plot.plot_raster(da=eggbox_vanilla)
rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)
sites_vanilla.plot(color="b", ax=axes)
for _k, row in sites_vanilla.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

## Helper functions

### Plotting

In [ ]:
def plot_profile_and_steep_segments(ray_profile, slope_critical):
    """Plot the elevation profile and its slopes along a ray, and marks steep parts."""
    r_m = ray_profile.r_m[:-1]
    z = ray_profile.z[:-1]
    dz = analyse.slopes(ray_profile)
    steep_mask = analyse.steep_mask(ray_profile, slope_critical=slope_critical)

    figure, axes = plt.subplots(1, 2, squeeze=True, figsize=(17, 7))
    axes[0].plot(r_m, z)
    axes[0].plot(r_m[steep_mask], z[steep_mask], c="r")
    axes[0].set_title(f"Profile along ray for {slope_critical=}")

    axes[1].plot(r_m, dz)
    axes[1].hlines(y=slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].hlines(y=-slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].set_title(f"Slope along ray for {slope_critical=}")

    return figure, axes

### Computation and evaluation of intermediate and final results

In [ ]:
def evaluate_rix_for_ray(ray_profile: RayProfile, slope_critical: float):
    """Evaluate RIX and related numbers."""
    n_steep_segments = analyse.steep_mask(ray_profile, slope_critical).sum()
    n_segments = analyse.slopes(ray_profile).size
    rix = RayResult(profile=ray_profile, slope_critical=slope_critical).ruggedness
    message = (
        f"Number of steep segments {n_steep_segments} vs total number of segments {n_segments} vs. RIX {rix:2.4f}."
    )
    return message

## Reprojections

A more detailed inspection of site 3:
- Ray facing south.
- Interpolation method:
  - nearest: Spiky as profile is step function. Resolution needs to be set properly.
  - linear: Unexpected changes in steepness of the profile leads to jumps in the slope.

In [ ]:
site = sites_utm.loc[3]
cfg = config.ANALYZER_DEFAULTS["parameters"]

LOCATION = site.geometry
THETA = 180
INTERPOLATION_METHOD = "linear"


ray = RayGeometry.from_compass_regular(location=LOCATION, theta=THETA, R_km=13, dr_km=0.5, crs=sites_utm.crs)
display("Coordinate arrays:")
display(ray.xs, ray.ys)
elevation_sampler = RegularGridXYSampler(da=eggbox_gcs, method=INTERPOLATION_METHOD)
ray_profile = RayProfile.create_regular(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.MASK)
display("RayProfile (regular)")
display(ray_profile.z)


RayResult(profile=ray_profile, slope_critical=0.013).describe()
ys = RayResult(profile=ray_profile, slope_critical=0.013).steep_segments_geodataframe()
print(np.array(ys[1:]) - np.array(ys[:-1]), min(ys), max(ys))

ys = RayResult(profile=ray_profile, slope_critical=0.013).steep_segments_geodataframe()
print(np.array(ys[1:]) - np.array(ys[:-1]), min(ys), max(ys))

_ = plot_profile_and_steep_segments(ray_profile, 0.013)
ys